# 🥈 Silver Layer: Data Cleaning & Identity Synthesis

This notebook demonstrates the transition from raw **Bronze** logs to high-quality **Silver** tables. We will apply the **Write-Audit-Publish (WAP)** pattern using Project Nessie.

### Objectives:
1. Read from the `bronze` branch.
2. Standardize schema and rename columns for clarity.
3. Apply window-based deduplication.
4. Synthesize a Customer Master table from order identities.
5. Publish results to the `main` branch via Nessie Merge.

## 🚀 Step 1: Configuration & Branch Setup
Crucially, we initialize our session on the `silver` branch to isolate our work.

In [ ]:
import pyspark
from pyspark.sql import SparkSession
import os

# Define Dependencies (Fix for S3FileIO)
PACKAGES = (
    "org.apache.iceberg:iceberg-spark-runtime-3.3_2.12:1.3.1,"
    "org.projectnessie.nessie-integrations:nessie-spark-extensions-3.3_2.12:0.67.0,"
    "software.amazon.awssdk:bundle:2.17.178,"
    "software.amazon.awssdk:url-connection-client:2.17.178,"
    "org.apache.hadoop:hadoop-aws:3.3.1"
)

# Oracle Object Storage Config
conf = (
    pyspark.SparkConf()
        .setAppName('Silver-Layer-Explorer')
        .set('spark.jars.packages', PACKAGES)
        .set('spark.sql.extensions', 'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,org.projectnessie.spark.extensions.NessieSparkSessionExtensions')
        .set('spark.sql.catalog.nessie', 'org.apache.iceberg.spark.SparkCatalog')
        .set('spark.sql.catalog.nessie.uri', "http://140.238.224.207:19120/api/v1")
        .set('spark.sql.catalog.nessie.ref', 'main')
        .set('spark.sql.catalog.nessie.authentication.type', 'NONE')
        .set('spark.sql.catalog.nessie.catalog-impl', 'org.apache.iceberg.nessie.NessieCatalog')
        .set('spark.sql.catalog.nessie.warehouse', "s3a://lakehouse-prod/warehouse")
        .set('spark.sql.catalog.nessie.io-impl', 'org.apache.iceberg.aws.s3.S3FileIO') 
        .set('spark.sql.catalog.nessie.s3.endpoint', "https://bmcfe6z38foz.compat.objectstorage.ap-mumbai-1.oraclecloud.com")
        .set('spark.hadoop.fs.s3a.access.key', "962c9f862226831e4edea90cfcfafb8a8dffcd51") # Oracle Key
        .set('spark.hadoop.fs.s3a.secret.key', "sd2rGU918DTmn35E4xJ8EV7BX2XUt7DkqC8v6WDNDUw=") # Oracle Secret
        .set('spark.hadoop.fs.s3a.endpoint', "https://bmcfe6z38foz.compat.objectstorage.ap-mumbai-1.oraclecloud.com")
        .set('spark.hadoop.fs.s3a.path.style.access', 'true')
        .set('spark.hadoop.fs.s3a.connection.ssl.enabled', 'true')
        .set('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
)

spark = SparkSession.builder.config(conf=conf).getOrCreate()

# Verify Configuration
print(f"✅ Spark Configured for S3FileIO")

## 📖 Step 2: Read from Bronze (Cross-Branch)
We use the `@bronze` syntax to explicitly pull raw data from the ingestion branch.

In [ ]:
bronze_df = spark.table("nessie.ecommerce.`orders_bronze@bronze` ")
print(f"📊 Bronze Row Count: {bronze_df.count():,}")

## 🔧 Step 3: Transformation Logic (Orders)
We standardize names and apply partitioning.

In [ ]:
def transform_batch(df):
    # 1. Rename and Cast
    standardized = df.withColumn("customer_id", F.col("user_id").cast("long")) \
                    .withColumn("order_date", F.to_date(F.col("event_time"))) \
                    .withColumn("processed_at", F.current_timestamp()) \
                    .drop("user_id")
    
    # 2. Deduplicate using Window function
    window_spec = Window.partitionBy("customer_id", "product_id", "event_time", "event_type").orderBy("event_time")
    deduped = standardized.withColumn("row_num", F.row_number().over(window_spec)) \
                          .filter("row_num == 1") \
                          .drop("row_num")
    
    return deduped

silver_orders = transform_batch(bronze_df)
print("✅ Transformation Logic Defined")

## 💾 Step 4: Write to Orders Silver
We use partitioned Iceberg tables.

In [ ]:
silver_orders.writeTo("nessie.ecommerce.orders_silver") \
    .using("iceberg") \
    .partitionedBy("order_date") \
    .createOrReplace()

print("💾 Orders Silver Table Created on branch 'silver'")

## 👥 Step 5: Synthetic Customer Identity Extraction
Extracting unique identities to fill the missing customer data gap.

In [ ]:
unique_users = silver_orders.select("customer_id").distinct()

synthetic_customers = unique_users \
    .withColumn("name", F.concat(F.lit("Customer_"), F.col("customer_id"))) \
    .withColumn("email", F.concat(F.col("name"), F.lit("@example.com"))) \
    .withColumn("signup_date", F.to_date(F.lit("2019-01-01")))

synthetic_customers.writeTo("nessie.ecommerce.customers_silver") \
    .using("iceberg") \
    .createOrReplace()

print(f"✅ Generated {synthetic_customers.count():,} Customer Identities")

## 🌿 Step 6: Publish to Production (WAP Pattern)
Finally, we merge the validated `silver` branch into `main`.

In [ ]:
spark.sql("MERGE BRANCH silver INTO main IN nessie")
print("🏁 PRODUCTION UPDATE: Silver is now LIVE on 'main'")

In [ ]:
# 1. Count Records (Should be ~300.3 Million)
df_silver = spark.table("nessie.ecommerce.orders_silver")
print(f"✅ Silver Table Count: {df_silver.count():,}")

# 2. Schema Verification (Check for 'order_date' and 'customer_id')
print("📋 Silver Schema:")
df_silver.printSchema()

# 3. Sample Data
print("📊 Sample Data:")
df_silver.show(5)